In [ ]:
# 1. Importações básicas
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Guardar gráficos em ficheiro (útil para colar no Word/LaTeX)
pio.defaults.default_format = "png"
pio.defaults.default_scale = 2

# 2. Leitura dos CSV (ajusta o caminho se necessário)
logs = pd.read_csv("logs_atendimento_real_tese.csv")
kpi_cenario = pd.read_csv("kpi_por_cenario.csv")
kpi_prioridade = pd.read_csv("kpi_por_prioridade.csv")
kpi_pontuacao = pd.read_csv("kpi_por_pontuacao_prioridade.csv")

# Conferir cabeçalhos
display(logs.head())
display(kpi_cenario.head())
display(kpi_prioridade.head())
display(kpi_pontuacao.head())


# 3. Funções para reforçar leitura visual dos gráficos
def reforco_visual_barras(fig, casas=2, texto_pos="outside"):
    fig.update_traces(
        texttemplate=f"%{{y:.{casas}f}}",
        textposition=texto_pos,
        cliponaxis=False,
        hovertemplate=f"%{{x}}<br>%{{fullData.name}}: %{{y:.{casas}f}}<extra></extra>",
    )
    fig.update_layout(uniformtext_minsize=9, uniformtext_mode="hide")
    fig.update_yaxes(tickformat=f".{casas}f")


def reforco_visual_pizza(fig, casas=2):
    fig.update_traces(
        textposition="inside",
        textinfo="label+percent+value",
        hovertemplate=f"%{{label}}<br>Valor: %{{value:.{casas}f}}<br>Percentual: %{{percent}}<extra></extra>",
    )


def reforco_visual_distribuicao(fig, eixo_x, eixo_y, casas=2):
    fig.update_traces(
        hovertemplate=f"{eixo_x}: %{{x}}<br>{eixo_y}: %{{y:.{casas}f}}<extra></extra>"
    )
    fig.update_yaxes(tickformat=f".{casas}f")


In [ ]:
# 2.x – Considerar apenas registos concluídos
logs_done = logs[logs["estado_final"] == "done"].copy()

# Converter colunas de tempo para datetime, se forem strings
for col in ["instante_entrada", "instante_chamada", "instante_fim"]:
    logs_done[col] = pd.to_datetime(logs_done[col])

# Calcular espera e atendimento em segundos
logs_done["espera_seg_calc"] = (logs_done["instante_chamada"] - logs_done["instante_entrada"]).dt.total_seconds()
logs_done["atendimento_seg_calc"] = (logs_done["instante_fim"] - logs_done["instante_chamada"]).dt.total_seconds()

# Converter para minutos
logs_done["espera_min_calc"] = logs_done["espera_seg_calc"] / 60.0
logs_done["atendimento_min_calc"] = logs_done["atendimento_seg_calc"] / 60.0

# Espreitar
logs_done[["cenario", "tipo_atendimento", "espera_min_calc", "atendimento_min_calc"]].head()


In [ ]:
kpi_from_logs_min = logs_done.groupby("cenario").agg(
    total=("id_item_fila", "count"),
    espera_media_min=("espera_min_calc", "mean"),
    espera_mediana_min=("espera_min_calc", "median"),
    espera_p95_min=("espera_min_calc", lambda x: x.quantile(0.95)),
    atendimento_media_min=("atendimento_min_calc", "mean"),
    atendimento_p95_min=("atendimento_min_calc", lambda x: x.quantile(0.95)),
).reset_index()

display(kpi_from_logs_min)


In [ ]:
# Bar empilhado: concluídos vs cancelados
fig1 = px.bar(
    kpi_cenario,
    x="cenario",
    y=["concluidos", "cancelados"],
    barmode="stack",
    title="Distribuição de estados finais por cenário",
    labels={"value": "Número de atendimentos", "variable": "Estado final"}
)
reforco_visual_barras(fig1, texto_pos="inside")
fig1.show()
fig1.write_image("fig4_1_estados_finais_stack.png")


In [ ]:
fig2 = px.pie(
    kpi_cenario,
    names="cenario",
    values="total",
    title="Proporção de atendimentos por cenário"
)
reforco_visual_pizza(fig2)
fig2.show()
fig2.write_image("fig4_2_pizza_total_cenario.png")


In [ ]:
kpi_cenario["taxa_autenticacao_pct"] = kpi_cenario["taxa_sucesso_autenticacao"] * 100

fig3 = px.bar(
    kpi_cenario,
    x="cenario",
    y="taxa_autenticacao_pct",
    title="Taxa de sucesso de autenticação biométrica por cenário",
    labels={"taxa_autenticacao_pct": "Taxa de sucesso (%)"}
)
fig3.update_yaxes(range=[0, 100])
reforco_visual_barras(fig3)
fig3.show()
fig3.write_image("fig4_3_taxa_autenticacao_pct.png")


In [ ]:
fig4 = px.bar(
    kpi_cenario,
    x="cenario",
    y=["espera_media_min", "espera_mediana_min", "espera_p95_min"],
    barmode="group",
    title="Estatísticas de tempo de espera por cenário (minutos)",
    labels={"value": "Tempo de espera (min)", "variable": "Estatística"}
)
reforco_visual_barras(fig4)
fig4.show()
fig4.write_image("fig4_4_espera_min_por_cenario.png")


In [ ]:
fig5 = px.bar(
    kpi_cenario,
    x="cenario",
    y=["atendimento_media_min", "atendimento_p95_min"],
    barmode="group",
    title="Estatísticas de tempo de atendimento por cenário (minutos)",
    labels={"value": "Tempo de atendimento (min)", "variable": "Estatística"}
)
reforco_visual_barras(fig5)
fig5.show()
fig5.write_image("fig4_5_atendimento_min_por_cenario.png")


In [ ]:
fig6 = px.box(
    logs_done,
    x="cenario",
    y="espera_min_calc",
    points="all",
    title="Distribuição dos tempos de espera por cenário (minutos)",
    labels={"espera_min_calc": "Tempo de espera (min)", "cenario": "Cenário"}
)
reforco_visual_distribuicao(fig6, eixo_x="Cenário", eixo_y="Espera (min)")
fig6.show()
fig6.write_image("fig4_6_box_espera_min_cenario.png")


In [ ]:
fig7 = px.violin(
    logs_done,
    x="cenario",
    y="espera_min_calc",
    box=True,
    points="all",
    title="Distribuição dos tempos de espera por cenário (violino, min)",
    labels={"espera_min_calc": "Tempo de espera (min)", "cenario": "Cenário"}
)
reforco_visual_distribuicao(fig7, eixo_x="Cenário", eixo_y="Espera (min)")
fig7.show()
fig7.write_image("fig4_7_violin_espera_min_cenario.png")


In [ ]:
fig8 = px.bar(
    kpi_prioridade,
    x="tipo_atendimento",
    y="espera_media_min",
    color="cenario",
    barmode="group",
    title="Tempo médio de espera por tipo de atendimento e cenário (min)",
    labels={"espera_media_min": "Tempo médio de espera (min)", "tipo_atendimento": "Tipo de atendimento"}
)
reforco_visual_barras(fig8)
fig8.show()
fig8.write_image("fig4_8_espera_media_min_tipo_cenario.png")


In [ ]:
fig9 = px.bar(
    kpi_prioridade,
    x="tipo_atendimento",
    y="espera_p95_min",
    color="cenario",
    barmode="group",
    title="Percentil 95 de espera por tipo de atendimento e cenário (min)",
    labels={"espera_p95_min": "P95 de espera (min)", "tipo_atendimento": "Tipo de atendimento"}
)
reforco_visual_barras(fig9)
fig9.show()
fig9.write_image("fig4_9_p95_espera_min_tipo_cenario.png")


In [ ]:
fig10 = px.bar(
    kpi_prioridade,
    x="tipo_atendimento",
    y="atendimento_media_min",
    color="cenario",
    barmode="group",
    title="Tempo médio de atendimento por tipo de atendimento e cenário (min)",
    labels={"atendimento_media_min": "Tempo médio de atendimento (min)", "tipo_atendimento": "Tipo de atendimento"}
)
reforco_visual_barras(fig10)
fig10.show()
fig10.write_image("fig4_10_atendimento_medio_min_tipo_cenario.png")


In [ ]:
fig11 = px.box(
    logs_done,
    x="tipo_atendimento",
    y="espera_min_calc",
    color="cenario",
    points="all",
    title="Distribuição dos tempos de espera por tipo de atendimento e cenário (min)",
    labels={"espera_min_calc": "Tempo de espera (min)", "tipo_atendimento": "Tipo de atendimento"}
)
reforco_visual_distribuicao(fig11, eixo_x="Tipo de atendimento", eixo_y="Espera (min)")
fig11.show()
fig11.write_image("fig4_11_box_espera_min_tipo_cenario.png")


In [ ]:
kpi_cenario["taxa_conclusao_pct"] = (kpi_cenario["concluidos"] / kpi_cenario["total"]) * 100

fig12 = px.bar(
    kpi_cenario,
    x="cenario",
    y="taxa_conclusao_pct",
    title="Taxa de conclusão de atendimentos por cenário",
    labels={"taxa_conclusao_pct": "Taxa de conclusão (%)"}
)
fig12.update_yaxes(range=[0, 100])
reforco_visual_barras(fig12)
fig12.show()
fig12.write_image("fig4_12_taxa_conclusao_pct.png")


In [ ]:
totais_estado = pd.DataFrame({
    "estado": ["concluidos", "cancelados"],
    "valor": [kpi_cenario["concluidos"].sum(), kpi_cenario["cancelados"].sum()]
})

fig13 = px.pie(
    totais_estado,
    names="estado",
    values="valor",
    title="Distribuição global de estados finais"
)
reforco_visual_pizza(fig13)
fig13.show()
fig13.write_image("fig4_13_pizza_estados_globais.png")


In [ ]:
fig14 = px.bar(
    kpi_pontuacao,
    x="faixa_prioridade",
    y="espera_media_min",
    color="cenario",
    barmode="group",
    title="Tempo médio de espera por faixa de pontuação de prioridade e cenário (min)",
    labels={"espera_media_min": "Tempo médio de espera (min)", "faixa_prioridade": "Faixa de prioridade"}
)
reforco_visual_barras(fig14)
fig14.show()
fig14.write_image("fig4_14_espera_media_min_faixa_prioridade.png")


In [ ]:
fig15 = px.bar(
    kpi_pontuacao,
    x="faixa_prioridade",
    y="espera_p95_min",
    color="cenario",
    barmode="group",
    title="Percentil 95 de espera por faixa de pontuação de prioridade e cenário (min)",
    labels={"espera_p95_min": "P95 de espera (min)", "faixa_prioridade": "Faixa de prioridade"}
)
reforco_visual_barras(fig15)
fig15.show()
fig15.write_image("fig4_15_p95_espera_min_faixa_prioridade.png")


| Secção | Conteúdo                                       | Gráficos sugeridos                            |
| ------ | ---------------------------------------------- | --------------------------------------------- |
| 4.2.1  | Desempenho global por cenário                  | fig1, fig2, fig3, fig4                        |
| 4.2.2  | Por tipo (normal/prioritário/urgente)          | fig5, fig6, fig7 (e fig8 se quiseres facetas) |
| 4.2.3  | Estados finais (done/cancelled)                | fig9, fig10                                   |
| 4.3    | Discussão das hipóteses (efeito da prioridade) | fig5–fig7, fig11, fig12                       |